# Emissions Analysis
Run the three cells in order: **1** upload the tool zip, **2** upload your Excel files (statistics, tables and charts appear below), **3** download all results.

In [ ]:
# CELL 1 - run once per session: upload emissions_tool.zip when asked
import os, sys, zipfile, subprocess
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not os.path.exists("emissions_tool"):
    if IN_COLAB:
        print("Choose emissions_tool.zip ...")
        up = files.upload()
        for name in up:
            if name.endswith(".zip"):
                zipfile.ZipFile(name).extractall(".")
    else:
        raise SystemExit("Put the emissions_tool folder next to this notebook.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xlrd", "openpyxl", "matplotlib"], check=False)
sys.path.insert(0, "emissions_tool")
print("Ready. Now run Cell 2.")


In [ ]:
# CELL 2 - upload your Excel / CSV files and get all results
import os, glob, shutil, sys
sys.path.insert(0, "emissions_tool")
import batch

# ---------------- settings you may change ----------------
EXCLUDE_SIZES   = ("Large",)   # sizes left out of the SME analysis; use () to keep everything
MISSING_AS_ZERO = True         # True: N/A / blank / "-" count as 0 (as in the report); False: leave them out
IQR_K           = 1.5          # outlier rule multiplier
# ----------------------------------------------------------

os.makedirs("input", exist_ok=True)
if IN_COLAB:
    for old in glob.glob("input/*"): os.remove(old)
    print("Choose one or many Excel/CSV files ...")
    up = files.upload()
    for name, content in up.items():
        open(os.path.join("input", name), "wb").write(content)
paths = sorted(p for p in glob.glob("input/*") if p.lower().endswith((".xlsx", ".xlsm", ".xls", ".csv")))
print(f"{len(paths)} file(s):", [os.path.basename(p) for p in paths], "\n")

out = batch.run(paths, "results", exclude_sizes=EXCLUDE_SIZES, missing_as_zero=MISSING_AS_ZERO, iqr_k=IQR_K)

from IPython.display import Image, display
for f in out["files"]:
    if f.endswith(".png"):
        display(Image(f))


In [ ]:
# CELL 3 - download everything (Excel results with tables and charts, chart images, key findings)
import shutil
shutil.make_archive("results", "zip", "results")
if IN_COLAB:
    files.download("results.zip")
else:
    print("Results are in the 'results' folder and results.zip")
